# 02 - Cleaning + reconciliation

Fixes found in 01:
1. duplicate tickets (helpdesk + legacy_fd copy) -> keep helpdesk
2. legacy_fd refund amounts are in paise -> /100
3. legacy resolved_at maybe UTC (policy s9)? -> checked, it's already IST in this export, no shift
4. missing order_id -> regex from message, then customer_id + sku fallback

Then: reconcile raw total -> clean total, rupee by rupee.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

raw = pd.read_csv('tickets.csv')
agents = pd.read_csv('agents.csv')
orders = pd.read_csv('orders.csv')
products = pd.read_csv('products.csv')

GOODWILL_CAP = 500      # policy s5
REPL_SHIPPING = 340     # policy s5, reverse pickup + forward shipping

len(raw)

12238

## Step 1 - remove duplicates

In [2]:
# helpdesk is the live system and stores rupees, so that copy wins
raw['_keep_rank'] = (raw.source_system != 'helpdesk').astype(int)
df = (raw.sort_values(['ticket_id', '_keep_rank'])
         .drop_duplicates('ticket_id', keep='first')
         .drop(columns='_keep_rank')
         .reset_index(drop=True))

dropped = raw.loc[~raw.index.isin(raw.sort_values(['ticket_id', '_keep_rank'])
                                  .drop_duplicates('ticket_id').index)]
raw = raw.drop(columns='_keep_rank')

print('kept:', len(df), '| dropped:', len(dropped))
print(dropped.source_system.value_counts())
assert df.ticket_id.is_unique

kept: 11600 | dropped: 638
source_system
legacy_fd    638
Name: count, dtype: int64


## Step 2 - paise -> rupees for legacy rows

In [3]:
is_legacy = df.source_system == 'legacy_fd'
df['refund_inr'] = np.where(is_legacy, df.refund_amount_inr / 100, df.refund_amount_inr)

# sanity: legacy and helpdesk refunds should now look alike
df.groupby('source_system').refund_inr.describe()

,count,mean,std,min,25%,50%,75%,max
source_system,,,,,,,,
helpdesk,1690.0,2850.817160,2135.919450,32.0,1299.00,2499.0,3499.0,13998.0
legacy_fd,650.0,2910.847692,2374.964166,57.0,1234.25,2249.0,4499.0,13998.0


## Step 3 - timestamps

Policy s9: legacy resolved_at was reconstructed from the event log, which stores UTC. README though says the export shows timestamps "as displayed in the helpdesk (IST)". Need to check which one is true before shifting anything.

In [4]:
for c in ['created_at', 'first_response_at', 'resolved_at']:
    df[c] = pd.to_datetime(df[c])

# test 1: if resolved_at were UTC (5.5h behind), some would fall before first response
print('resolved before first response:')
print((df.resolved_at < df.first_response_at).groupby(df.source_system).sum())

resolved before first response:
source_system
helpdesk     0
legacy_fd    0
dtype: int64


In [5]:
# test 2: hour-of-day of resolution + handle time. UTC would shift legacy peak hours by ~5.5h
done = df[df.resolved_at.notna()]
hours = pd.crosstab(done.resolved_at.dt.hour, done.source_system, normalize='columns').mul(100).round(1)
handle_hrs = (done.resolved_at - done.first_response_at).dt.total_seconds() / 3600

print('median handle time (hrs):')
print(handle_hrs.groupby(done.source_system).median().round(2))
hours.T

median handle time (hrs):
source_system
helpdesk     0.45
legacy_fd    0.45
dtype: float64


resolved_at,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
source_system,,,,,,,,,,,,,,,,,,,,,,,,
helpdesk,3.1,2.3,1.7,1.5,1.3,1.0,1.1,1.8,2.3,3.8,4.8,5.5,6.6,6.5,5.8,6.0,4.9,4.9,5.8,6.2,6.3,5.9,6.0,4.6
legacy_fd,3.3,2.5,1.6,1.6,0.8,1.2,0.8,1.6,2.7,3.6,5.4,5.2,6.8,6.2,6.5,5.2,5.1,5.2,5.7,5.8,7.1,5.8,6.0,4.5


Same hour pattern and same median handle time for both systems -> legacy resolved_at is already IST in this export. **Decision: no shift.** Adding 5:30 would make every legacy ticket look 5.5h slower.

In [6]:
# refund month = when it was resolved (money goes out), fallback created_at for open/pending
df['refund_month'] = df.resolved_at.fillna(df.created_at).dt.to_period('M')
df['created_month'] = df.created_at.dt.to_period('M')

r = df[df.refund_inr.notna()]
moved = r[r.refund_month != r.created_month]
print('refunds whose month changes:', len(moved), '| value: Rs', f'{moved.refund_inr.sum():,.0f}')
print('refunds with no resolved_at (using created):', r.resolved_at.isna().sum())

refunds whose month changes: 80 | value: Rs 243,212
refunds with no resolved_at (using created): 115


## Step 4 - recover order_id

In [7]:
df['order_src'] = np.where(df.order_id.notna(), 'field', None)

# a) customer typed it in the message
from_msg = df.customer_message.str.extract(r'(VR\d{6})', flags=2)[0].str.upper()
fill_a = df.order_id.isna() & from_msg.isin(orders.order_id)
df.loc[fill_a, 'order_id'] = from_msg[fill_a]
df.loc[fill_a, 'order_src'] = 'message'

# b) README fallback: customer_id + sku, only when exactly one order matches
one_match = orders.groupby(['customer_id', 'sku']).filter(lambda g: len(g) == 1)
lookup = one_match.set_index(['customer_id', 'sku']).order_id
keys = pd.MultiIndex.from_arrays([df.customer_id, df.product_sku])
from_join = pd.Series(lookup.reindex(keys).values, index=df.index)
fill_b = df.order_id.isna() & from_join.notna()
df.loc[fill_b, 'order_id'] = from_join[fill_b]
df.loc[fill_b, 'order_src'] = 'cust+sku'

df.order_src.fillna('missing').value_counts()

order_src
field       7701
cust+sku    3032
missing      655
message      212
Name: count, dtype: int64

## Step 5 - join agents + products

In [8]:
# roster is one row per assignment -> pick the row valid on the ticket date
ag = agents.copy()
ag['from_date'] = pd.to_datetime(ag.from_date)
ag['to_date'] = pd.to_datetime(ag.to_date).fillna(pd.Timestamp('2099-12-31'))

m = df[['ticket_id', 'agent_id', 'created_at']].merge(ag, on='agent_id', how='left')
m = m[(m.created_at >= m.from_date) & (m.created_at <= m.to_date)]
df = df.merge(m[['ticket_id', 'name', 'team', 'site', 'tier']]
                .rename(columns={'name': 'agent_name', 'team': 'agent_team'}),
              on='ticket_id', how='left')
print('tickets without a valid roster row:', df.agent_team.isna().sum())

df = df.merge(products[['sku', 'product_name', 'unit_cost_inr']],
              left_on='product_sku', right_on='sku', how='left').drop(columns='sku')
df.shape

tickets without a valid roster row: 0


(11600, 31)

## Step 6 - policy flags

In [9]:
has_refund = df.refund_inr.notna()

df['flag_gw_over_cap'] = has_refund & (df.refund_reason_code == 'GW-OTHER') & (df.refund_inr > GOODWILL_CAP)
df['flag_refund_and_repl'] = has_refund & (df.replacement_issued == 'Y')

# customer was owed ONE remedy, so only the cheaper one counts as the extra cost (conservative)
repl_cost = df.unit_cost_inr + REPL_SHIPPING
df['double_payout_leak'] = np.where(df.flag_refund_and_repl, np.minimum(df.refund_inr, repl_cost), 0)
df['double_payout_upper'] = np.where(df.flag_refund_and_repl, df.refund_inr + repl_cost, 0)

df[['flag_gw_over_cap', 'flag_refund_and_repl']].sum()

flag_gw_over_cap        879
flag_refund_and_repl    166
dtype: int64

In [10]:
q = df.refund_month.dt.asfreq('Q')
pd.DataFrame({
    'gw_over_cap_rs': df[df.flag_gw_over_cap].groupby(q).refund_inr.sum(),
    'double_payouts': df[df.flag_refund_and_repl].groupby(q).size(),
    'leak_conservative': df[df.flag_refund_and_repl].groupby(q).double_payout_leak.sum(),
    'leak_upper': df[df.flag_refund_and_repl].groupby(q).double_payout_upper.sum(),
}).fillna(0).astype(int)

,gw_over_cap_rs,double_payouts,leak_conservative,leak_upper
refund_month,,,,
2025Q1,229530,15,29280,92864
2025Q2,277766,14,22040,64934
2025Q3,490026,24,42270,121397
2025Q4,776961,38,73680,223725
2026Q1,515259,42,70651,201450
2026Q2,564119,33,57220,166791
2026Q3,17971,0,0,0


## Step 7 - reconciliation waterfall

Every rupee of Arjun's raw sum has to land in exactly one bucket.

In [11]:
raw_total = raw.refund_amount_inr.sum()
dup_removed = dropped.refund_amount_inr.sum()
legacy_kept_raw = df.loc[is_legacy, 'refund_amount_inr'].sum()
paise_fix = legacy_kept_raw - legacy_kept_raw / 100
clean_total = df.refund_inr.sum()

waterfall = pd.DataFrame({
    'step': ['Raw export total (as summed)', 'less: duplicate re-import rows',
             'less: legacy paise -> rupees', 'True refund total'],
    'rs': [raw_total, -dup_removed, -paise_fix, clean_total],
})

# the key check
assert abs(raw_total - dup_removed - paise_fix - clean_total) < 0.01

waterfall.assign(rs=waterfall.rs.map('{:,.0f}'.format))

,step,rs
0,Raw export total (as summed),"230,124,081"
1,less: duplicate re-import rows,"-36,101,100"
2,less: legacy paise -> rupees,"-187,313,049"
3,True refund total,"6,709,932"


In [12]:
# monthly true refunds (refund month basis) - compare with helpdesk's ~11L/quarter
monthly = df[has_refund].groupby('refund_month').refund_inr.agg(['count', 'sum'])
print(monthly.groupby(monthly.index.asfreq('Q'))['sum'].sum().map('{:,.0f}'.format))
assert abs(monthly['sum'].sum() - clean_total) < 0.01
monthly

refund_month
2025Q1      603,085
2025Q2      724,023
2025Q3    1,169,173
2025Q4    1,665,693
2026Q1    1,260,872
2026Q2    1,258,324
2026Q3       28,762
Freq: Q-DEC, Name: sum, dtype: object


,count,sum
refund_month,,
2025-01,58,141048.0
2025-02,62,176319.0
2025-03,90,285718.0
2025-04,78,244365.0
2025-05,88,227787.0
2025-06,85,251871.0
2025-07,116,341066.0
2025-08,129,384316.0
2025-09,154,443791.0


## Save

In [13]:
os.makedirs('output', exist_ok=True)
df.to_csv('output/tickets_clean.csv', index=False)
waterfall.to_csv('output/waterfall.csv', index=False)
print('saved', df.shape)

saved (11600, 35)


## Notes / decisions

- **Duplicates:** kept the helpdesk copy (live system, already rupees). 638 legacy rows dropped.
- **Paise:** legacy refund / 100. Proven by 125 pairs with ratio exactly 100.
- **Timestamps:** policy says the legacy event log is UTC, but in this export legacy resolved_at has the same hour pattern and handle time as helpdesk -> already IST. Did NOT shift. Would have added 5.5h to every legacy ticket.
- **Refund month:** month of resolved_at (when money goes out), created_at if not resolved yet (115 refunds). 80 refunds (Rs 2.4L) land in a different month vs using created_at.
- **July 2026:** 10 refunds (Rs 28.7K) resolved after the data cut-off. Kept them and will label July as partial so the total still reconciles.
- **order_id:** message regex first, then customer+sku only if exactly one order matches (avoid guessing). 655 still missing, fine for now since refunds don't need order_id.
- **Double payout cost:** customer was owed one remedy, so the leak = the cheaper of (refund) or (unit cost + 340). Full cost kept only as an upper bound.
- **Reconciliation:** 23.01 cr - 3.61 cr dups - 18.73 cr paise = 67.1 L. Assert passes.